# 第 1 章互动实验：像素与色彩

这不是作业答案，而是一张可操纵的实验台。目标是先**看到现象**，再回到 `.py` 作业独立实现。

建议顺序：

1. 每个代码单元先预测结果，再运行。
2. 修改参数，至少制造一次失败案例。
3. 把观察写入 `实验记录.md`。
4. 不要把这里使用 OpenCV 基线的代码复制到“从零实现”作业。

## 0. 环境与数据

本单元先找课程根目录下的 `images/test_underexposed.jpg` 和 `images/test_color.jpg`。这两张图不在仓库里，要用的话先在课程根目录运行 `python scripts/generate_sample_images.py`。找不到时会打印一行说明，然后用代码生成内容相同的两张图，后面的单元照常运行。

本单元还定义了 `load_homework` 和 `attempt`，第 5 节用它们调用你在作业文件里写的函数；作业还没完成时只打印提示，不会报错。

In [ ]:
%matplotlib inline
import importlib.util
import os
from pathlib import Path

import cv2
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, IntSlider

np.random.seed(42)
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.sans-serif'] = ['Microsoft YaHei', 'SimHei', 'PingFang SC', 'Heiti SC', 'Noto Sans CJK SC', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False


def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError('请在课程目录内启动 Jupyter')


CHAPTER = find_chapter('01_像素与色彩')


def load_homework(filename):
    """读入本章的一个作业文件。每次调用都重新读，改完作业后重新运行单元就能看到新结果。
    设置环境变量 COURSE_IMPL_DIR 可以换成另一份实现（维护者验证用）。"""
    override = os.environ.get('COURSE_IMPL_DIR')
    base = CHAPTER
    if override:
        base = Path(override) / CHAPTER.name
        if not base.is_dir():                      # 相对路径：按课程根目录再找一次
            base = CHAPTER.parent / override / CHAPTER.name
    spec = importlib.util.spec_from_file_location('homework_' + filename[:3].replace('-', '_'), base / filename)
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    return module


def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，不中断 Notebook。"""
    try:
        result = fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f'先完成作业里的 {fn.__name__}：{exc}')
        return None
    if result is None:
        print(f'先完成作业里的 {fn.__name__}（现在返回的是 None）')
    return result


COURSE_ROOT = CHAPTER.parent


def read_image(path, flags):
    """文件不存在时返回 None。先读成字节再解码，路径里有中文时在 Windows 上也能读。"""
    if not path.exists():
        return None
    return cv2.imdecode(np.fromfile(str(path), dtype=np.uint8), flags)


GRAY_PATH = COURSE_ROOT / 'images' / 'test_underexposed.jpg'
COLOR_PATH = COURSE_ROOT / 'images' / 'test_color.jpg'
gray = read_image(GRAY_PATH, cv2.IMREAD_GRAYSCALE)
color_bgr = read_image(COLOR_PATH, cv2.IMREAD_COLOR)
if gray is None or color_bgr is None:
    print('没有找到 images/ 下的测试图，改用代码生成的两张图（内容和 scripts/generate_sample_images.py 生成的相同，只是没有经过 JPEG 压缩）。')
    rng = np.random.default_rng(42)
    gray = np.zeros((400, 500), dtype=np.uint8)
    gray[:300, :300] = rng.integers(20, 60, (300, 300), dtype=np.uint8)
    gray[100:300, 200:500] = rng.integers(160, 220, (200, 300), dtype=np.uint8)
    gray[50:100, 350:450] = 240
    for x in range(30, 500, 100):
        cv2.line(gray, (x, 30), (x, 370), 200, 2)
    color_bgr = np.zeros((300, 500, 3), dtype=np.uint8)
    color_bgr[:, :166, 2] = 255
    color_bgr[:, 167:333, 1] = 255
    color_bgr[:, 334:, 0] = 255
    color_bgr[50:250, 50:450] = np.linspace(0, 255, 400, dtype=np.uint8)[None, :, None]

print('OpenCV:', cv2.__version__)
print('灰度图:', gray.shape, gray.dtype, (int(gray.min()), int(gray.max())))
print('彩色图:', color_bgr.shape, color_bgr.dtype)

## 1. 图像是带 dtype 的张量

先预测下面两个结果：

- `uint8(200) + uint8(100)` 是 300、255，还是 44？
- 转成整数后再相加，会发生什么？

图像处理中的许多“诡异黑点”其实不是算法问题，而是数据类型溢出。

In [ ]:
a = np.array([200], dtype=np.uint8)
b = np.array([100], dtype=np.uint8)

print('uint8 直接相加:', a + b)
print('转成 int16 相加:', a.astype(np.int16) + b.astype(np.int16))
print('饱和到 0-255:', np.clip(a.astype(np.int16) + b.astype(np.int16), 0, 255).astype(np.uint8))

### 小实验：亮度增益与截断

拖动增益。观察：

- 均值如何变化？
- 饱和到 255 的像素比例何时快速上升？
- 亮度增强为什么不能恢复已经饱和的细节？

In [ ]:
def show_gain(gain=1.0):
    enhanced = np.clip(gray.astype(np.float32) * gain, 0, 255).astype(np.uint8)
    saturated = np.mean(enhanced == 255)
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(enhanced, cmap='gray', vmin=0, vmax=255)
    axes[0].set_title(f'gain={gain:.2f}, 饱和比例={saturated:.2%}')
    axes[0].axis('off')
    axes[1].hist(enhanced.ravel(), bins=256, range=(0, 256), color='black')
    axes[1].set_xlim(0, 255)
    axes[1].set_title('灰度直方图')
    plt.show()

interact(show_gain, gain=FloatSlider(min=0.3, max=3.0, step=0.1, value=1.0));

## 2. BGR、RGB 与 HSV

OpenCV 读入 BGR，Matplotlib 按 RGB 显示。如果直接 `plt.imshow(color_bgr)`，红蓝会交换。

下面同时显示：错误显示、正确显示、三个单通道。注意“某个区域在某通道更亮”意味着什么。

In [ ]:
color_rgb = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2RGB)
b, g, r = cv2.split(color_bgr)

fig, axes = plt.subplots(1, 5, figsize=(16, 3.5))
items = [
    (color_bgr, None, '错误：BGR 当 RGB'),
    (color_rgb, None, '正确：转成 RGB'),
    (b, 'gray', 'B 通道'),
    (g, 'gray', 'G 通道'),
    (r, 'gray', 'R 通道'),
]
for ax, (im, cmap, title) in zip(axes, items):
    ax.imshow(im, cmap=cmap, vmin=0 if cmap else None, vmax=255 if cmap else None)
    ax.set_title(title)
    ax.axis('off')
plt.tight_layout()

### 小实验：只修改 HSV 的 V 通道

拖动 `v_scale`。这是一种方便的亮度操作，但不是物理曝光模型：截断、色彩空间量化和相机响应曲线都会带来差异。

In [ ]:
def show_hsv_value_scale(v_scale=1.0):
    hsv = cv2.cvtColor(color_bgr, cv2.COLOR_BGR2HSV).astype(np.float32)
    hsv[:, :, 2] = np.clip(hsv[:, :, 2] * v_scale, 0, 255)
    result_bgr = cv2.cvtColor(hsv.astype(np.uint8), cv2.COLOR_HSV2BGR)
    result_rgb = cv2.cvtColor(result_bgr, cv2.COLOR_BGR2RGB)

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    axes[0].imshow(color_rgb)
    axes[0].set_title('原图')
    axes[1].imshow(result_rgb)
    axes[1].set_title(f'V × {v_scale:.2f}')
    for ax in axes: ax.axis('off')
    plt.show()

interact(show_hsv_value_scale, v_scale=FloatSlider(min=0.3, max=2.5, step=0.1, value=1.0));

## 3. 直方图与 CDF

这一步只用 OpenCV 的实现作为**基线观察**。你的作业仍需独立实现 CDF 查表。

比较原图和均衡化图：均衡化通常增大对比度，但不保证视觉上更自然，也可能放大平坦暗区里的噪声。

In [ ]:
equalized_cv = cv2.equalizeHist(gray)

fig, axes = plt.subplots(2, 2, figsize=(11, 8))
axes[0, 0].imshow(gray, cmap='gray', vmin=0, vmax=255)
axes[0, 0].set_title('原图')
axes[0, 1].imshow(equalized_cv, cmap='gray', vmin=0, vmax=255)
axes[0, 1].set_title('OpenCV 均衡化基线')
axes[1, 0].hist(gray.ravel(), bins=256, range=(0, 256), color='black')
axes[1, 0].set_title('原图直方图')
axes[1, 1].hist(equalized_cv.ravel(), bins=256, range=(0, 256), color='black')
axes[1, 1].set_title('均衡化后直方图')
for ax in axes[0]: ax.axis('off')
plt.tight_layout()

hist = np.bincount(gray.ravel(), minlength=256)
cdf = hist.cumsum() / hist.sum()
print('CDF 单调不减:', bool(np.all(np.diff(cdf) >= 0)))
print('CDF 末值:', cdf[-1])

### 思考

1. 为什么均衡化后的直方图并不会每个 bin 一样高？
2. 如果图像只有 4 个离散灰度值，映射后最多能有多少个灰度值？
3. CDF 中的零计数区间为什么需要特殊处理？

把答案写入实验记录，再去完成 `1-1_直方图均衡化.py`。

## 4. 噪声与滤波的匹配

下面控制椒盐噪声比例和中值窗口大小。请重点观察两件事：

- 噪声比例过高时，局部窗口的中位数也可能被污染。
- 窗口越大不一定越好：细线、小角点和纹理会被消除。

In [ ]:
def salt_pepper_for_demo(img, amount, seed=42):
    rng = np.random.default_rng(seed)
    noisy = img.copy()
    random_map = rng.random(img.shape)
    noisy[random_map < amount / 2] = 0
    noisy[(random_map >= amount / 2) & (random_map < amount)] = 255
    return noisy


def show_noise_filter(amount=0.05, kernel=3):
    if kernel % 2 == 0:
        kernel += 1
    noisy = salt_pepper_for_demo(gray, amount)
    median = cv2.medianBlur(noisy, kernel)
    mean = cv2.blur(noisy, (kernel, kernel))

    fig, axes = plt.subplots(1, 4, figsize=(16, 4))
    for ax, im, title in zip(
        axes,
        [gray, noisy, median, mean],
        ['原图', f'椒盐 {amount:.0%}', f'中值 {kernel}×{kernel}', f'均值 {kernel}×{kernel}'],
    ):
        ax.imshow(im, cmap='gray', vmin=0, vmax=255)
        ax.set_title(title)
        ax.axis('off')
    plt.tight_layout()
    plt.show()

interact(
    show_noise_filter,
    amount=FloatSlider(min=0.0, max=0.30, step=0.01, value=0.05),
    kernel=IntSlider(min=3, max=11, step=2, value=3),
);

## 5. 讲义新增内容的三个实验

下面三个实验对应讲义 1.1、1.3、1.4 节新增的实测内容，也是讲义 1.9 节前三个破坏实验。每个实验先在 `实验记录.md` 里写下预测，再运行代码单元。

实验 B、C 调用你在作业文件里写的函数。作业还没完成时，单元只打印一行“先完成作业里的 …”，不会报错；完成作业后重新运行即可，不用重启内核。

### 实验 A：两张 `uint8` 图相减的三种写法（讲义 1.1 节，图 1）

参考图和待检图是同一张灰度 120 的平坦图，各自加了标准差 3 的噪声。待检图上另有一个比周围亮 30 的 4×4 缺陷和一个比周围暗 30 的 4×4 缺陷。差值大于 20 的像素算超限。

先预测：

- `np.abs(test - ref)`、`cv2.subtract(test, ref)`、`cv2.absdiff(test, ref)` 三种写法，各有多少像素超限？全图共 16384 个像素。
- 哪一种写法看不见暗缺陷？
- 哪一种写法的错最难被发现？

In [ ]:
rng = np.random.default_rng(1)
ref = np.clip(np.round(120 + rng.normal(0, 3, (128, 128))), 0, 255).astype(np.uint8)
test_float = 120 + rng.normal(0, 3, (128, 128))
test_float[40:44, 40:44] += 30      # 亮缺陷
test_float[80:84, 80:84] -= 30      # 暗缺陷
test = np.clip(np.round(test_float), 0, 255).astype(np.uint8)

ways = {
    'np.abs(test - ref)': np.abs(test - ref),
    'cv2.subtract(test, ref)': cv2.subtract(test, ref),
    'cv2.absdiff(test, ref)': cv2.absdiff(test, ref),
    '先转 int16 再相减取绝对值': np.abs(test.astype(np.int16) - ref.astype(np.int16)),
}
print(f"{'写法':<28}{'超限像素':>8}{'亮缺陷内超限':>10}{'暗缺陷内超限':>10}{'差值最大值':>8}")
for name, diff in ways.items():
    over = diff > 20
    print(f'{name:<30}{int(over.sum()):>8}{int(over[40:44, 40:44].sum()):>12}{int(over[80:84, 80:84].sum()):>12}{int(diff.max()):>10}')

fig, axes = plt.subplots(1, 4, figsize=(15, 3.6))
for ax, (name, diff) in zip(axes, ways.items()):
    ax.imshow(diff, cmap='gray', vmin=0, vmax=60)
    ax.set_title(name, fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('两个缺陷各有 16 个像素。对照讲义 1.1 节的表：哪一行“两个缺陷都检出”其实没有意义？')

### 实验 B：均衡化有没有让弱缺陷更容易检出（讲义 1.3 节，图 3）

一张 256×256 的图：背景灰度 100，高斯噪声标准差 2，中间有一块 24×24 的区域比背景亮 5。用你写的 `histogram_equalization` 处理它。对比噪声比的定义见讲义 1.2 节：缺陷均值与背景均值之差的绝对值，除以背景的标准差。

先预测三件事：

- 均衡化前后，最高的那根直方图柱一样高吗？
- 用到的灰度级个数变多还是变少？
- 那一小块的灰度差会被放大。它的对比噪声比是升、是降，还是不变？和线性拉伸相比呢？

In [ ]:
def contrast_to_noise(img, defect_mask):
    background = img[~defect_mask].astype(np.float64)
    return abs(img[defect_mask].mean() - background.mean()) / background.std(), background.std()

rng = np.random.default_rng(3)
scene = 100 + rng.normal(0, 2, (256, 256))
defect = np.zeros(scene.shape, dtype=bool)
defect[116:140, 116:140] = True
scene[defect] += 5
low_contrast = np.clip(np.round(scene), 0, 255).astype(np.uint8)

lo, hi = int(low_contrast.min()), int(low_contrast.max())
stretched = np.round((low_contrast.astype(np.float64) - lo) * 255 / (hi - lo)).astype(np.uint8)
versions = {'原图': low_contrast, '线性拉伸到 0 到 255': stretched}

equalize = load_homework('1-1_直方图均衡化.py').histogram_equalization
mine = attempt(equalize, low_contrast)
if mine is not None:
    versions['你的均衡化'] = mine
    print('你的结果与 cv2.equalizeHist 的最大差：', int(np.abs(mine.astype(int) - cv2.equalizeHist(low_contrast).astype(int)).max()))

print(f"{'处理':<16}{'背景噪声':>8}{'灰度差':>8}{'对比噪声比':>8}{'最高的柱':>8}{'灰度级个数':>8}")
for name, img in versions.items():
    cnr, noise = contrast_to_noise(img, defect)
    hist = np.bincount(img.ravel(), minlength=256)
    gap = abs(img[defect].mean() - img[~defect].mean())
    print(f'{name:<18}{noise:>10.2f}{gap:>10.1f}{cnr:>12.2f}{int(hist.max()):>12}{int(np.count_nonzero(hist)):>12}')

fig, axes = plt.subplots(1, len(versions), figsize=(4.6 * len(versions), 4))
for ax, (name, img) in zip(np.atleast_1d(axes), versions.items()):
    ax.imshow(img, cmap='gray', vmin=0, vmax=255)
    ax.set_title(name)
    ax.axis('off')
plt.tight_layout()
plt.show()
print('画面上哪一张的缺陷最醒目？表里哪一行的对比噪声比最高？这两个答案一样吗？')

### 实验 C：20% 的椒盐，中值滤波之后还剩多少坏点（讲义 1.4 节，图 4）

一张 256×256、灰度 120 的平坦图，用你写的 `add_salt_pepper_noise` 加 20% 的椒盐，再做 3×3 和 5×5 的中值滤波（为了快，这一步用 `cv2.medianBlur`）。滤波后不等于 120 的像素就是残留的坏点，图像内部和最外一圈分开数。

先预测：

- 加噪后椒和盐各占多少？合计是不是正好 20% 左右？
- 3×3 中值之后，内部残留的坏点比例大约是多少？5×5 呢？
- 最外一圈的残留比例比内部高还是低，差多少倍？为什么？

最后一段把你写的 `median_filter_manual` 和 `cv2.medianBlur` 在一小块图上做全图比较，包括最外一圈。

In [ ]:
homework = load_homework('1-3_椒盐噪声与中值滤波.py')
flat = np.full((256, 256), 120, dtype=np.uint8)
np.random.seed(0)
noisy = attempt(homework.add_salt_pepper_noise, flat, 0.20)
if noisy is not None and np.array_equal(noisy, flat):
    print('先完成作业里的 add_salt_pepper_noise（现在加噪后的图和原图相同）')
    noisy = None

if noisy is not None:
    pepper, salt = np.mean(noisy == 0), np.mean(noisy == 255)
    print(f'加噪后：椒 {pepper:.3f}，盐 {salt:.3f}，合计 {pepper + salt:.3f}（设定值 0.200）')
    ring = np.ones(flat.shape, dtype=bool)
    ring[1:-1, 1:-1] = False
    print(f"{'窗口':<8}{'内部残留比例':>10}{'最外一圈残留比例':>12}{'全图残留坏点数':>10}")
    for k in (3, 5):
        bad = cv2.medianBlur(noisy, k) != 120
        print(f'{k}×{k:<6}{bad[~ring].mean():>14.2e}{bad[ring].mean():>18.2e}{int(bad.sum()):>14}')
    print('对照讲义 1.4 节的表（20% 一列）。这里只有一张图，数字会有起伏，量级应该一致。')

    small = noisy[:48, :48]
    mine = attempt(homework.median_filter_manual, small, 3)
    if mine is not None and not np.any(mine):
        print('先完成作业里的 median_filter_manual（现在输出全是 0）')
    elif mine is not None:
        diff = mine.astype(int) != cv2.medianBlur(small, 3).astype(int)
        inner = diff[1:-1, 1:-1].sum()
        print(f'你的 3×3 中值与 cv2.medianBlur 在 48×48 的图上不同的像素：内部 {int(inner)} 个，最外一圈 {int(diff.sum() - inner)} 个（都应为 0）')

## 6. 破坏实验

至少完成下面两个：

1. 把噪声比例提高到 20%，比较 3×3 与 9×9 中值滤波。
2. 用带细文字的真实图片测试大窗口滤波，记录细节如何消失。
3. 对高斯噪声使用中值滤波，再与高斯滤波比较。
4. 对低对比度但噪声很强的图片做全局均衡化，观察噪声是否被放大。

不要只保存“最好看”的结果。失败案例更能暴露方法的假设。

## 7. 离开 Notebook 完成作业

现在关闭提示，独立完成：

- `1-1_直方图均衡化.py`
- `1-2_通道分离与伪彩色.py`
- `1-3_椒盐噪声与中值滤波.py`

完成后运行：

```bash
python tests/check_progress.py
```

最后填写 `实验记录.md`。自动测试通过只是底线；你还需要能够解释每个结果为什么发生。